Make NumPy available:

In [3]:
import numpy as np

## Exercise 07.1 (indexing and timing)

Create two very long NumPy 1D-arrays (vectors) `x` and `y` and sum the arrays using:

1. The NumPy addition syntax, `z = x + y`; and
2. A `for` loop that computes the sum entry-by-entry

Compare the time required for the two approaches for arrays of different lengths (use a very long array for 
the timing). The values of the array entries are not important for this test. Use `%time` to report the time.

*Hint:* To loop over an array using indices, try a construction like:

In [4]:
x = np.ones(10)
y = np.ones(len(x))
for i in range(len(x)):
    print(x[i]*y[i])

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


#### (1) Add two arrays using built-in addition operator:

In [5]:
x1 = np.random.rand(10)
y1 = np.random.rand(10)
x2 = np.random.rand(10000000)
y2 = np.random.rand(10000000)
%time z1 = x1 + y1
%time z2 = x2 + y2

CPU times: user 78 μs, sys: 8 μs, total: 86 μs
Wall time: 320 μs
CPU times: user 14.1 ms, sys: 12.4 ms, total: 26.5 ms
Wall time: 26.6 ms


#### (2) Add two arrays using own implementation:

In [7]:
def addition(x, y):
    z = np.zeros(len(x))
    for i in range(len(x)):
        z[i] = x[i] + y[i]
    return z

x1 = np.random.rand(10)
y1 = np.random.rand(10)
x2 = np.random.rand(10000000)
y2 = np.random.rand(10000000)
%time z = addition(x1, y1)
%time z = addition(x2, y2)

CPU times: user 41 μs, sys: 5 μs, total: 46 μs
Wall time: 48.2 μs
CPU times: user 1.1 s, sys: 2.82 ms, total: 1.1 s
Wall time: 1.1 s


### Optional extension: just-in-time (JIT) compilation

You will see a large difference in the time required between your NumPy and 'plain' Python implementations. This is due to Python being an *interpreted* language as opposed to a *compiled* language. A way to speed up plain Python implementions is to convert the interpreted Python code into compiled code. A tool for doing this is [Numba](https://numba.pydata.org/).

Below is an example using Numba and JIT to accelerate a computation:

In [9]:
%pip -q install numba 
import numba
import math

def compute_sine_native(x):
    z = np.zeros(len(x))
    for i in range(len(z)):
        z[i] = math.sin(x[i])
    return z

@numba.jit
def compute_sine_jit(x):
    z = np.zeros(len(x))
    for i in range(len(z)):
        z[i] = math.sin(x[i])
    return z
    
x = np.ones(10000000)
%time z = compute_sine_native(x)
compute_sine_jit(x)
%time z = compute_sine_jit(x)

Note: you may need to restart the kernel to use updated packages.
CPU times: user 796 ms, sys: 15.4 ms, total: 811 ms
Wall time: 811 ms
CPU times: user 31.6 ms, sys: 24.2 ms, total: 55.8 ms
Wall time: 55.8 ms


**Task:** Test if Numba can be used to accelerate your implementation that uses indexing to sum two arrays, and by how much.

In [13]:
def addition_native(x, y):
    z = np.zeros(len(x))
    for i in range(len(x)):
        z[i] = x[i] + y[i]
    return z

@numba.jit
def addition_jit(x, y):
    z = np.zeros(len(x))
    for i in range(len(x)):
        z[i] = x[i] + y[i]
    return z

x = np.random.rand(10000000)
y = np.random.rand(10000000)
%time z = addition_native(x, y)
%time z = addition_jit(x, y)

# 11 times faste after implementing jit

CPU times: user 1.12 s, sys: 7.44 ms, total: 1.13 s
Wall time: 1.13 s
CPU times: user 64.8 ms, sys: 15.8 ms, total: 80.6 ms
Wall time: 81.4 ms


## Exercise 07.2 (member functions and slicing)

Anonymised scores (out of 60) for an examination are stored in a NumPy array. Write:

1. A function that takes a NumPy array of the raw scores and returns the scores as percentages, sorted from 
   lowest to highest (try using `scores.sort()`, where `scores` is a NumPy array holding the scores).
1. A function that returns the maximum, minimum and mean of the raw scores as a dictionary with the 
   keys '`min`', '`max`' and '`mean`'. Use the NumPy array functions `min()`, `max()` and `mean()` to do the 
   computation, e.g. `max = scores.max()`.  
   
   Design your function for the min, max and mean to optionally exclude the highest and lowest scores from the 
   computation of the min, max and mean. 
   
   *Hint:* sort the array of scores and use array slicing to exclude
   the first and the last entries.

Use the scores 
```python
scores = np.array([58.0, 35.0, 24.0, 42, 7.8])
```
to test your functions.

In [33]:
def to_percentage_and_sort(scores):
    scores.sort()
    scores_percentage = scores*100/60
    return scores_percentage

def statistics(scores, exclude=False):
    result = {}
    newScores = np.sort(scores)
    if exclude:
        newScores = np.array(newScores)[1:-1]
    result["max"] = newScores.max()
    result["min"] = newScores.min()
    result["mean"] = newScores.mean()
    return result

In [34]:
## tests ##

scores = np.array([35.0, 58.0, 24.0, 42, 7.8])
assert np.isclose(to_percentage_and_sort(scores), [ 13.0, 40.0, 58.33333333,  70.0, 96.66666667]).all()

s0 = statistics(scores)
assert np.isclose(s0["min"], 7.8)
assert np.isclose(s0["mean"], 33.36)
assert np.isclose(s0["max"], 58.0)

s1 = statistics(scores, True)
assert np.isclose(s1["min"], 24.0)
assert np.isclose(s1["mean"], 33.666666666666666667)
assert np.isclose(s1["max"], 42.0)

## Exercise 07.3 (slicing)

For the two-dimensional array

In [35]:
A = np.array([[4.0, 7.0, -2.43, 67.1],
             [-4.0, 64.0, 54.7, -3.33],
             [2.43, 23.2, 3.64, 4.11],
             [1.2, 2.5, -113.2, 323.22]])
print(A)

[[   4.      7.     -2.43   67.1 ]
 [  -4.     64.     54.7    -3.33]
 [   2.43   23.2     3.64    4.11]
 [   1.2     2.5  -113.2   323.22]]


use array slicing for the below operations, printing the results to the screen to check. Try to use array slicing such that your code would still work if the dimensions of `A` were enlarged.

**Check your results carefully against hand computations.**

#### 1. Extract the third column as a 1D array

In [37]:
print(A[:, 2])

[  -2.43   54.7     3.64 -113.2 ]


#### 2. Extract the first two rows as a 2D sub-array

In [39]:
print(A[0:2])

[[ 4.    7.   -2.43 67.1 ]
 [-4.   64.   54.7  -3.33]]


#### 3.  Extract the bottom-right $2 \times 2$ block as a 2D sub-array

In [43]:
print(A[-2:,-2:])

[[   3.64    4.11]
 [-113.2   323.22]]


#### 4. Sum the last column

In [45]:
print(A[:, -1].sum())

391.1


#### Compute transpose

Compute the transpose of `A` (search online to find the function/syntax to do this).

In [46]:
print(A.T)

[[   4.     -4.      2.43    1.2 ]
 [   7.     64.     23.2     2.5 ]
 [  -2.43   54.7     3.64 -113.2 ]
 [  67.1    -3.33    4.11  323.22]]


## Exercise 07.4 (optional extension)

In a previous exercise (04.5) you implemented the bisection algorithm to find approximate roots of a mathematical function. Use the SciPy bisection function `optimize.bisect` (https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.bisect.html) to find roots of the mathematical function that was used in the previous exercise. Compare the results computed by SciPy and your program from the earlier exercise, and compare the computational time (using `%time`).

In [52]:
from scipy import optimize
def compute_root(f, x0, x1, tol, max_it):
    """Compute roots of a function using bisection"""
    it = 0
    error = 1 + tol
    x_mid = (x0 + x1) / 2
    while it < max_it and error > tol:
        if f(x_mid) * f(x0) < 0:
            x1 = x_mid
            x_mid = (x_mid + x0) / 2
        else:
            x0 = x_mid
            x_mid = (x_mid + x1) / 2
        it += 1
        error = abs(f(x_mid))
    f_mid = f(x_mid)
    return x_mid

def my_f2(x):
    return x**5 / 9 + x**4 + x**2 - 8 * x**2 + 3 * x + 7

%time x = compute_root(my_f2, x0=-1, x1=0, tol=1.0e-6, max_it=1000)
%time x1 = optimize.bisect(f=my_f2, a=-1, b=0, xtol=1.0e-6, maxiter=1000)

CPU times: user 25 μs, sys: 0 ns, total: 25 μs
Wall time: 25.7 μs
CPU times: user 52 μs, sys: 0 ns, total: 52 μs
Wall time: 54.1 μs
